In [1]:
import pandas as pd

search_df = pd.read_parquet("../data/raw_data/search_dataset.parquet")

In [2]:
search_df_model = search_df[
    (search_df["designer"].notna()) & 
    (search_df["keyword"].notna())
].copy()

# Feature exploration and engineering


In [3]:
#  user_id and keyword aren't enough to identify a query session
#  we utilize the index for a particular user_id x keyword pair to identify a cutoff point
#  that point allows us to use a query_id label that we can group by later on to identify and analyze unique query sessions
#  noting that a user_id x keyword pair can occur multiple times across different sessions (assumption: a user can run the same query multiple times) 
#  role of using ts here is to enable the split logic. For example, a user can query "Dior" at 10 AM and then later at 2 PM. The results should not be mixed, hence why we include it in the group by.
#  
search_df_model = search_df_model.sort_values(["user_id", "keyword", "ts", "index"])
search_df_model["index_reset"] = search_df_model.groupby(["user_id", "keyword"])["index"].diff() <= 0
search_df_model["index_reset"] = search_df_model["index_reset"].fillna(True)  # first row of each group starts a session
search_df_model["query_id"] = search_df_model.groupby(["user_id", "keyword"])["index_reset"].cumsum()

"""
In other words: within each user-keyword pair, monotonically increasing index values belong to the same search event, 
and any index that drops signals a new one.  The query_id counter labels which search event each row belongs to.
"""

query_signature_final_cols = ["user_id", "keyword", "query_id"]

In [4]:
import regex as re

def detect_script(text):
    if pd.isna(text):
        return 'unknown'
    
    text = str(text)
    has_arabic = bool(re.search(r'\p{Arabic}', text))
    has_latin = bool(re.search(r'\p{Latin}', text))
    
    if has_arabic and has_latin:
        return 'mixed'
    elif has_arabic:
        return 'arabic'
    elif has_latin:
        return 'latin'
    else:
        return 'other'


def match_substring_vec(col_a, col_b):
    a = col_a.fillna("").str.lower()
    b = col_b.fillna("").str.lower()
    return pd.Series(
        [kw in prod if kw != "" else False for kw, prod in zip(a, b)],
        index=col_a.index
    ).astype(int)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances
import numpy as np

# Fit on all product names + keywords combined
corpus = pd.concat([search_df_model["name"].fillna(""), search_df_model["keyword"].fillna("")]).unique()
tfidf = TfidfVectorizer(max_features=5000)
tfidf.fit(corpus)

# Transform and compute row-wise similarity
kw_vectors = tfidf.transform(search_df_model["keyword"].fillna(""))
name_vectors = tfidf.transform(search_df_model["name"].fillna(""))

# Row-wise cosine similarity (not full pairwise matrix)
search_df_model["tfidf_similarity"] = 1 - paired_cosine_distances(kw_vectors, name_vectors)


In [6]:
query_features = ["keyword_length","tfidf_similarity"]
product_features = ["product_ctr", "designer_ctr", "productclass_ctr", "class_ctr"]
interaction_features = ["kw_matches_designer", "kw_matches_class", "kw_matches_productclass", "kw_in_product_name"]
one_hot_features = ["lang_latin", "lang_mixed", "lang_other", "intent_navigational"]


In [7]:
search_df_model['keyword_lang'] = search_df_model['keyword'].apply(detect_script)
search_df_model['keyword_length'] = search_df_model['keyword'].str.len()
known_designers = set(search_df_model["designer"].dropna().str.lower().unique())
search_df_model["intent"] = search_df_model["keyword"].str.lower().apply(
    lambda x: "navigational" if any(d in x for d in known_designers) else "exploratory"
)

In [74]:
search_df_model["keyword_length"].value_counts()


keyword_length
5     828355
6     495077
4     487934
7     314503
11    277743
10    276622
9     244929
8     235765
3     235322
12    151561
13    103756
14     58525
15     48585
16     36115
17     32323
18     22389
19     14293
20      9200
2       8947
22      6184
21      5573
23      3909
24      2864
25      1624
1       1541
26      1442
29      1298
28      1097
27      1062
32       653
31       303
37       209
36        76
34        63
33        63
30        59
59        48
42        45
47        43
39        40
41        34
35        16
38        11
48        10
46         8
56         7
44         6
45         6
50         3
40         3
52         3
54         2
43         2
83         2
49         2
67         2
Name: count, dtype: int64

In [75]:
product_stats = search_df_model.groupby("product_id").agg(
    views=("plp_view", "sum"),
    clicks=("plp_click", "sum")
)
product_stats["product_ctr"] = product_stats["clicks"] / product_stats["views"]

designer_stats = search_df_model.groupby("designer").agg(
    views=("plp_view", "sum"),
    clicks=("plp_click", "sum")
)
designer_stats["designer_ctr"] = designer_stats["clicks"] / designer_stats["views"]

product_class_stats = search_df_model.groupby("productclass").agg(
    views=("plp_view", "sum"),
    clicks=("plp_click", "sum")
)
product_class_stats["productclass_ctr"] = product_class_stats["clicks"] / product_class_stats["views"]

class_stats = search_df_model.groupby("class").agg(
    views=("plp_view", "sum"),
    clicks=("plp_click", "sum")
)
class_stats["class_ctr"] = class_stats["clicks"] / class_stats["views"]

search_df_model = search_df_model.merge(
    product_stats[["product_ctr"]].reset_index(), 
    on="product_id", 
    how="left"
)

search_df_model = search_df_model.merge(
    designer_stats[["designer_ctr"]].reset_index(), 
    on="designer", 
    how="left"
)

search_df_model = search_df_model.merge(
    product_class_stats[["productclass_ctr"]].reset_index(), 
    on="productclass", 
    how="left"
)

search_df_model = search_df_model.merge(
    class_stats[["class_ctr"]].reset_index(), 
    on="class", 
    how="left"
)

In [76]:
kw = search_df_model["keyword"].str.lower()

search_df_model["kw_matches_designer"] = search_df_model["keyword"].str.lower().apply(
    lambda x: 1 if any(d in x for d in known_designers) else 0
)

search_df_model["kw_matches_class"] = match_substring_vec(kw, search_df_model["class"])
search_df_model["kw_matches_productclass"] = match_substring_vec(kw, search_df_model["productclass"])
search_df_model["kw_in_product_name"] = match_substring_vec(kw, search_df_model["name"])

In [77]:
lang_dummies = pd.get_dummies(search_df_model["keyword_lang"], prefix="lang", drop_first=True).astype(int)
intent_dummies = pd.get_dummies(search_df_model["intent"], prefix="intent", drop_first=True).astype(int)
search_df_model = pd.concat([search_df_model, lang_dummies, intent_dummies], axis=1)

In [78]:
# fill 28 productclass nulls
search_df_model["productclass_ctr"] = search_df_model.productclass_ctr.fillna(search_df_model.productclass_ctr.mean())

In [ ]:
search_df_model

,dt,ts,user_id,app_country,keyword,index,product_id,productclass,class,designer,...,productclass_ctr,class_ctr,kw_matches_designer,kw_matches_class,kw_matches_productclass,kw_in_product_name,lang_latin,lang_mixed,lang_other,intent_navigational
0,2025-08-01,2025-08-01 10:20:49,00006617A3D6DBEC8649E27874B5307170CF25B5A7E0A8...,Country_B,حذاء رياضيه,1,FE1405E3F18D326A96295B4EDCB549F70B635C25770861...,Shoes,Flats,Tory Burch,...,0.021225,0.017639,0,0,0,0,0,0,0,0
1,2025-08-01,2025-08-01 10:20:49,00006617A3D6DBEC8649E27874B5307170CF25B5A7E0A8...,Country_B,حذاء رياضيه,2,A747F9D5C97DC631268314DA0918FB2E80480D458F7402...,Shoes,Flats,Tory Burch,...,0.021225,0.017639,0,0,0,0,0,0,0,0
2,2025-08-01,2025-08-01 10:20:49,00006617A3D6DBEC8649E27874B5307170CF25B5A7E0A8...,Country_B,حذاء رياضيه,3,FCF295C9E126DD98AA0390D8069EB0FE21CA9A57CD7A5E...,Shoes,Flats,Tory Burch,...,0.021225,0.017639,0,0,0,0,0,0,0,0
3,2025-08-01,2025-08-01 10:20:49,00006617A3D6DBEC8649E27874B5307170CF25B5A7E0A8...,Country_B,حذاء رياضيه,4,B8F84D142342FF6291631B2F4E835066AE1B86A83EA7DB...,Shoes,Flats,Schutz,...,0.021225,0.017639,0,0,0,0,0,0,0,0
4,2025-08-01,2025-08-01 10:20:49,00006617A3D6DBEC8649E27874B5307170CF25B5A7E0A8...,Country_B,حذاء رياضيه,5,1DAF3184C9446B983207A923CDE96922F722377652ACDE...,Shoes,Flats,Tory Burch,...,0.021225,0.017639,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3910252,2025-08-01,2025-08-01 21:37:29,FFFD0D21D1DF900E8B65F0FD574403C800EE152B4E042F...,Country_C,Brown heels,12,A33F31EC34734B999568FDEB4CC472367ACC79A6614894...,Shoes,Heels,Malone Souliers,...,0.021225,0.015650,0,0,0,0,1,0,0,0
3910253,2025-08-01,2025-08-01 21:37:29,FFFD0D21D1DF900E8B65F0FD574403C800EE152B4E042F...,Country_C,Brown heels,13,3ECB62029D6EF5887A242A143D7CE12933C7DAADB2FE42...,Shoes,Heels,Balenciaga,...,0.021225,0.015650,0,0,0,0,1,0,0,0
3910254,2025-08-01,2025-08-01 21:37:29,FFFD0D21D1DF900E8B65F0FD574403C800EE152B4E042F...,Country_C,Brown heels,15,77167288B139865121F586C120A125DC5A57F8819C9CA0...,Shoes,Heels,Malone Souliers,...,0.021225,0.015650,0,0,0,0,1,0,0,0
3910255,2025-08-01,2025-08-01 21:37:29,FFFD0D21D1DF900E8B65F0FD574403C800EE152B4E042F...,Country_C,Brown heels,16,F0A55A6952D085D626E3DAACAC8B4C874E069EE8193BD2...,Shoes,Heels,Chloé,...,0.021225,0.015650,0,0,0,0,1,0,0,0


In [117]:
final_engineered_features = ["index"] + interaction_features + one_hot_features  #+ product_features + interaction_features + one_hot_features +

## Train test split

In [118]:
# OUTPUT CONTRACT:
# - Each row remains one product impression (unit of observation unchanged)
# - query_signature_final_cols = ["user_id", "keyword", "query_id"] uniquely identifies a query session
# - Within a session: rows share the same user, keyword, and search event; index is monotonically increasing
# - Across sessions: the model trains on impressions, evaluates on sessions (MRR averaged across sessions)
# - Granularity shift: model scores at impression level → groupby session for ranking → aggregate across sessions for metrics

In [119]:
query_signature_final_cols = ["user_id", "keyword", "query_id"]

In [128]:
# Get unique query sessions 
# query signature cols hold the 3 composite keys that represent a session

from sklearn.model_selection import train_test_split

query_sessions = search_df_model[query_signature_final_cols].drop_duplicates()

# Split at query level
train_sessions, test_sessions = train_test_split(query_sessions, test_size=0.05, random_state=42)

# Map back to rows
train_df = search_df_model.merge(train_sessions, on=query_signature_final_cols, how="inner")
test_df = search_df_model.merge(test_sessions, on=query_signature_final_cols, how="inner")

print(f"Train: {len(train_df)} rows, {len(train_sessions)} queries")
print(f"Test: {len(test_df)} rows, {len(test_sessions)} queries")

Train: 3717035 rows, 242485 queries
Test: 193222 rows, 12763 queries


In [129]:
train_df.columns

Index(['dt', 'ts', 'user_id', 'app_country', 'keyword', 'index', 'product_id',
       'productclass', 'class', 'designer', 'name', 'plp_view', 'plp_click',
       'is_converted', 'index_reset', 'query_id', 'tfidf_similarity',
       'keyword_lang', 'keyword_length', 'intent', 'product_ctr',
       'designer_ctr', 'productclass_ctr', 'class_ctr', 'kw_matches_designer',
       'kw_matches_class', 'kw_matches_productclass', 'kw_in_product_name',
       'lang_latin', 'lang_mixed', 'lang_other', 'intent_navigational'],
      dtype='str')

In [130]:
baseline_v1_train_model_data = train_df[final_engineered_features]
target_variables = train_df.plp_click.values


In [131]:
baseline_v1_test_model_data = test_df[final_engineered_features]
test_target_variables = test_df.plp_click.values


In [132]:
baseline_v1_test_model_data["index"] = baseline_v1_train_model_data["index"].mean() # Neutralize testing index

In [133]:
from sklearn.preprocessing import StandardScaler

bv1_scaler = StandardScaler()
baseline_v1_train_model_data_scaled = bv1_scaler.fit_transform(baseline_v1_train_model_data)


In [134]:
baseline_v1_test_model_data_scaled = bv1_scaler.transform(baseline_v1_test_model_data)

# Model

## baseline 

In [135]:
# to answer: In the current ranking system, where does the first clicked product sit, averaged across all test sessions?

In [136]:
# All test query sessions, not just clicked ones
all_test_queries = test_df[query_signature_final_cols].drop_duplicates()

# For each query session, rank by original index order
test_df["original_rank"] = test_df.groupby(query_signature_final_cols)["index"].rank(ascending=True, method="first")

# For each query session, find the rank of the first clicked product
# If no click in session, this query contributes 0 (via fillna)
first_click_rank = (
    test_df[test_df["plp_click"] == 1]
    .groupby(query_signature_final_cols)["original_rank"]
    .min()
    .reset_index()
    .rename(columns={"original_rank": "first_click_rank"})
)

# Merge back to all query sessions, not just clicked ones
baseline0_eval = all_test_queries.merge(first_click_rank, on=query_signature_final_cols, how="left")
baseline0_eval["reciprocal_rank"] = (1 / baseline0_eval["first_click_rank"]).fillna(0)

mrr_baseline0 = baseline0_eval["reciprocal_rank"].mean()
print(f"Baseline 0 MRR: {mrr_baseline0:.4f}")

Baseline 0 MRR: 0.2450


## Logistic Regression with Abalation for feature selection

In [ ]:
"""
Feature Ablation Experiment
Run every combination of feature groups to find what helps and what hurts.
Uses logistic regression for speed.
"""

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import pandas as pd
import numpy as np
import time

# --- Feature groups ---
feature_groups = {
    "tfidf":       ["tfidf_similarity"],
    "interaction":  ["kw_matches_designer", "kw_matches_class", "kw_matches_productclass", "kw_in_product_name"],
    "ctr":          ["product_ctr", "designer_ctr", "productclass_ctr", "class_ctr"],
    "lang":         ["lang_latin", "lang_mixed", "lang_other"],
    "intent":       ["intent_navigational"],
    "kw_length":    ["keyword_length"],
}

# Position is always included (neutralized at inference)
position_col = ["index"]

# --- MRR computation function ---
def compute_mrr(test_df, prob_col, query_sig_cols):
    all_sessions = test_df[query_sig_cols].drop_duplicates()
    test_df["_pred_rank"] = test_df.groupby(query_sig_cols)[prob_col].rank(ascending=False, method="first")
    
    first_click = (
        test_df[test_df["plp_click"] == 1]
        .groupby(query_sig_cols)["_pred_rank"]
        .min()
        .reset_index()
        .rename(columns={"_pred_rank": "first_click_rank"})
    )
    
    eval_df = all_sessions.merge(first_click, on=query_sig_cols, how="left")
    eval_df["rr"] = (1 / eval_df["first_click_rank"]).fillna(0)
    return eval_df["rr"].mean()

# --- Run all non-empty subsets of feature groups ---
results = []
group_names = list(feature_groups.keys())

for r in range(1, len(group_names) + 1):
    for combo in combinations(group_names, r):
        # Assemble feature list
        feature_cols = position_col.copy()
        for g in combo:
            feature_cols.extend(feature_groups[g])
        
        # Prepare data
        X_train = train_df[feature_cols].copy()
        X_test = test_df[feature_cols].copy()
        X_test["index"] = X_train["index"].mean()  # neutralize position
        
        # Scale
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train
        lr = LogisticRegression(class_weight="balanced", max_iter=1000, solver="lbfgs", C=1.0)
        t0 = time.time()
        lr.fit(X_train_scaled, train_df["plp_click"].values)
        train_time = time.time() - t0
        
        # Predict
        y_pred = lr.predict_proba(X_test_scaled)[:, 1]
        test_df["_click_prob"] = y_pred
        
        # Evaluate
        mrr = compute_mrr(test_df, "_click_prob", query_signature_final_cols)
        
        results.append({
            "features": " + ".join(combo),
            "n_features": len(feature_cols),
            "mrr": round(mrr, 4),
            "train_sec": round(train_time, 1)
        })
        
        print(f"  {' + '.join(combo):60s} → MRR: {mrr:.4f}  ({train_time:.1f}s)")

# Clean up temp columns
test_df.drop(columns=["_pred_rank", "_click_prob"], inplace=True, errors="ignore")

# --- Summary table ---
results_df = pd.DataFrame(results).sort_values("mrr", ascending=False).reset_index(drop=True)
print("\n" + "="*80)
print("FEATURE ABLATION RESULTS (sorted by MRR)")
print("="*80)
print(f"\nBaseline 0 (current system): 0.2462\n")
print(results_df.to_string(index=False))

In [88]:
results_df[:20]

,features,n_features,mrr,train_sec
0,lang + intent + kw_length,6,0.2432,0.8
1,intent,2,0.2432,0.7
2,lang + kw_length,5,0.2432,0.8
3,kw_length,2,0.2432,0.9
4,lang + intent,5,0.2432,0.9
5,lang,4,0.2432,0.7
6,intent + kw_length,3,0.2432,0.8
7,interaction + lang + intent + kw_length,10,0.2320,0.9
8,interaction + lang + kw_length,9,0.2320,1.1
9,interaction + intent + kw_length,7,0.2318,0.9


## LR

In [137]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [138]:
lr = LogisticRegression(
    class_weight="balanced",      # compensate for click sparsity (~2% CTR)
    max_iter=1000,                 # default 100 often doesn't converge on large data
    solver="lbfgs",               # default, works well for this scale
    C=1.0)                        # default regularization, no reason to tune for a baseline

lr.fit(baseline_v1_train_model_data_scaled,target_variables)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [139]:
y_pred = lr.predict_proba(baseline_v1_test_model_data_scaled)[:,1]

In [140]:
y_pred

array([0.5016986 , 0.5016986 , 0.5016986 , ..., 0.49736808, 0.54887103,
       0.54887103], shape=(193222,))

In [141]:
test_df["click_probability"] = y_pred

In [142]:
# All test query sessions, not just clicked ones
all_test_queries = test_df[query_signature_final_cols].drop_duplicates()

# For each query session, rank by original index order
test_df["predicted_rank"] = test_df.groupby(query_signature_final_cols)["click_probability"].rank(ascending=False, method="first")

# For each query session, find the rank of the first clicked product
# If no click in session, this query contributes 0 (via fillna)
model_first_click_rank = (
    test_df[test_df["plp_click"] == 1]
    .groupby(query_signature_final_cols)["predicted_rank"]
    .min()
    .reset_index()
    .rename(columns={"predicted_rank": "model_first_click_rank"})
)

# Merge back to all query sessions, not just clicked ones
baselinev1_eval = all_test_queries.merge(model_first_click_rank, on=query_signature_final_cols, how="left")
baselinev1_eval["reciprocal_rank"] = (1 / baselinev1_eval["model_first_click_rank"]).fillna(0)

mrr_baseline0 = baselinev1_eval["reciprocal_rank"].mean()
print(f"Baseline 1 MRR: {mrr_baseline0:.4f}")

Baseline 1 MRR: 0.2331


## SVM

In [127]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# LinearSVC doesn't have predict_proba, so wrap it
svc = LinearSVC(class_weight="balanced", max_iter=1000)
model_svm = CalibratedClassifierCV(svc, cv=3)


model_svm.fit(baseline_v1_train_model_data_scaled,target_variables)
y_pred_svm = model_svm.predict_proba(baseline_v1_test_model_data_scaled)[:, 1]

test_df["click_probability"] = y_pred_svm

test_df["predicted_rank"] = test_df.groupby(query_signature_final_cols)["click_probability"].rank(ascending=False, method="first")

all_test_queries = test_df[query_signature_final_cols].drop_duplicates()
first_click = (
    test_df[test_df["plp_click"] == 1]
    .groupby(query_signature_final_cols)["predicted_rank"]
    .min().reset_index().rename(columns={"predicted_rank": "first_click_rank"})
)
eval_df = all_test_queries.merge(first_click, on=query_signature_final_cols, how="left")
eval_df["rr"] = (1 / eval_df["first_click_rank"]).fillna(0)
print(f"SVM MRR: {eval_df['rr'].mean():.4f}")

SVM MRR: 0.2318


## LGBM

In [106]:
import os
os.environ.pop("MPLBACKEND", None)

import matplotlib
matplotlib.use("agg")

import lightgbm as lgb

# Train
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=50,       # approximate ratio of negatives to positives
    min_child_samples=100,     # prevent overfitting on sparse click signal
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

model.fit(baseline_v1_train_model_data, target_variables)



,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,100


In [107]:
y_pred = model.predict_proba(baseline_v1_test_model_data)[:, 1]

In [108]:
y_pred

array([0.33782257, 0.33782257, 0.33782257, ..., 0.33782257, 0.33782257,
       0.33782257], shape=(389516,))

In [109]:
test_df["click_probability"] = y_pred

In [110]:
# All test query sessions, not just clicked ones
all_test_queries = test_df[query_signature_final_cols].drop_duplicates()

# For each query session, rank by original index order
test_df["predicted_rank"] = test_df.groupby(query_signature_final_cols)["click_probability"].rank(ascending=False, method="first")

# For each query session, find the rank of the first clicked product
# If no click in session, this query contributes 0 (via fillna)
model_first_click_rank = (
    test_df[test_df["plp_click"] == 1]
    .groupby(query_signature_final_cols)["predicted_rank"]
    .min()
    .reset_index()
    .rename(columns={"predicted_rank": "model_first_click_rank"})
)

# Merge back to all query sessions, not just clicked ones
baselinev1_eval = all_test_queries.merge(model_first_click_rank, on=query_signature_final_cols, how="left")
baselinev1_eval["reciprocal_rank"] = (1 / baselinev1_eval["model_first_click_rank"]).fillna(0)

mrr_baseline0 = baselinev1_eval["reciprocal_rank"].mean()
print(f"Baseline 1 MRR: {mrr_baseline0:.4f}")

Baseline 1 MRR: 0.2432
